## Pretext Tasks
### Relative positioning

In [1]:
import numpy as np
from PIL import Image, ImageDraw

# play with variables to understand better
PATCH_SIZE = 96
GAP = 3

def make_example(img):
    drawer = ImageDraw.Draw(img) 
    image = np.array(img) # convert the image into an numpy array

    # compute offsets so that center patch remains within image bounds.
    offset_x, offset_y = image.shape[0] - (PATCH_SIZE*3 + GAP*2), image.shape[1] - (PATCH_SIZE*3 + GAP*2)
    # select the  starting coordinate of the center patch, within offsets
    start_grid_x, start_grid_y = np.random.randint(0, offset_x), np.random.randint(0, offset_y)
    # select randomly among 8 spatial arrangements relative to center patch
    patch_loc_arr = [(1, 1), (2, 1), (3, 1), (1, 2), (3, 2), (1, 3), (2, 3), (3, 3)]
    loc = np.random.randint(len(patch_loc_arr))
    tempx, tempy = patch_loc_arr[loc]
    
    # compute x and y coordinate of random patch depending on the spatial configuration selected
    # use GAP to inducle gap between patches and avoid trivial solutions
    patch_x_pt = start_grid_x + PATCH_SIZE * (tempx-1) + GAP * (tempx-1)
    patch_y_pt = start_grid_y + PATCH_SIZE * (tempy-1) + GAP * (tempy-1)
    # crop the random patch and plot on image
    random_patch = image[patch_x_pt:patch_x_pt+PATCH_SIZE, patch_y_pt:patch_y_pt+PATCH_SIZE]
    drawer.rectangle([(patch_x_pt, patch_y_pt), (patch_x_pt + PATCH_SIZE, patch_y_pt + PATCH_SIZE) ], outline ="red", width=4)

    # compute the x and y coordinate of center patch
    patch_x_pt = start_grid_x + PATCH_SIZE + GAP 
    patch_y_pt = start_grid_y + PATCH_SIZE  + GAP
    # crop the center patch and plot
    center_patch = image[patch_x_pt:patch_x_pt+PATCH_SIZE, patch_y_pt:patch_y_pt+PATCH_SIZE]
    drawer.rectangle([(patch_x_pt, patch_y_pt), (patch_x_pt + PATCH_SIZE, patch_y_pt + PATCH_SIZE) ], outline ="blue", width=4)

    random_patch_label = loc
    # return center patch, random patch and the classification label
    return center_patch, random_patch, random_patch_label, img

image = Image.open("n02107683_Bernese_mountain_dog.jpeg")
center_patch, random_patch, random_patch_label, patched_image = make_example(image)
patched_image.save("output/image.png")
print("Pseudo Label is: ", random_patch_label)

FileNotFoundError: [Errno 2] No such file or directory: 'n02107683_Bernese_mountain_dog.jpeg'

In [ ]:
import torch.nn as nn
import torch

class AlexNetwork(nn.Module):
  def __init__(self):
      super(AlexNetwork, self).__init__()
      # block containing convolutions, relu and normalization layers
      self.cnn = nn.Sequential(
        nn.Conv2d(3, 96, kernel_size=11, stride=4),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2),
        nn.LocalResponseNorm(96),
        
        nn.Conv2d(96, 384, kernel_size=5, stride = 2,padding = 2),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2),
        nn.LocalResponseNorm(384),
        
        nn.Conv2d(384, 384, kernel_size=3, stride=1,padding = 1),
        nn.ReLU(inplace=True),
        nn.BatchNorm2d(384),
        
        nn.Conv2d(384, 384, kernel_size=3, stride=1,padding = 1),
        nn.ReLU(inplace=True),
        nn.BatchNorm2d(384),
        
        nn.Conv2d(384, 256, kernel_size=3, stride=1,padding = 1),
        nn.ReLU(inplace=True),
        nn.BatchNorm2d(256),
        nn.MaxPool2d(kernel_size=3, stride=2,padding = 1),
      )
      # an fc layer to project the convolutional embedding to 4096 dimensional vector
      self.fc6 = nn.Sequential(
        nn.Linear(256,4096),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(4096),
      )
      
      # a classifier that takes concatenated center and random patch features
      # classifies the concatenated feature into 8 classes
      self.classifier = nn.Sequential(
        nn.Linear(2*4096,4096), #fc7
        nn.ReLU(inplace=True),

        nn.Linear(4096, 4096), #fc8
        nn.ReLU(inplace=True),

        nn.Linear(4096, 8) #fc9
      )

  def forward_once(self, x):
    output= self.cnn(x)
    output = output.view(output.size()[0], -1)
    output = self.fc6(output)
    return output


  def forward(self, center_patch, random_patch):
    output_fc_center = self.forward_once(center_patch) # compute representation of center patch
    output_fc_random = self.forward_once(random_patch) # compute representation of random patch
    output = torch.cat((output_fc_center, output_fc_random), 1) # concatenate output_fc_center and output_fc_random 
    output = self.classifier(output) # return logits corresponding to 8 classes
    return output

if __name__ == '__main__':
  model = AlexNetwork()
  center_patch = torch.randn((8,3,96,96))
  random_patch = torch.randn((8,3,96,96))
  logits = model(center_patch, random_patch)
  predictions = torch.argmax(logits, 1)
  for i in range(8):
    print("Network prediction for {}-th pair : ".format(i), predictions[i].item())


### Image rotation

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

# utility function to show images in a grid.
def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size
    
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid

def make_example(img):
    # resize image to 224 x 224 resolution
    img = img.resize((224,224))
    draw = ImageDraw.Draw(img)

    # rotate by 90 degrees
    img1 = img.rotate(90)
    draw1 = ImageDraw.Draw(img1)
    
    # rotate by 180 degrees
    img2 = img.rotate(180)
    draw2 = ImageDraw.Draw(img2)

    # rotate by 270 degrees
    img3 = img.rotate(270)
    draw3 = ImageDraw.Draw(img3)

    # assign labels to each of them.
    draw.text((0, 0),"Label: 0",(255,255,255))
    draw1.text((0, 0),"Label: 1",(255,255,255))
    draw2.text((0, 0),"Label: 2",(255,255,255))
    draw3.text((0, 0),"Label: 3",(255,255,255))

    return [img, img1, img2, img3], [0, 1, 2, 3]

image = Image.open("n02107683_Bernese_mountain_dog.jpeg")
images, labels = make_example(image)
final_image = image_grid(images, 1, 4)
final_image.save("output/images.png")
print("Pseudo labels are: ", labels)


In [ ]:
import torch.nn as nn
import torch
import torchvision
from torchvision.models import resnet18
from PIL import Image
import torchvision.transforms.functional as T

# rotate images
def make_example(img):
    img = img.resize((224,224))
    img1 = img.rotate(90)
    img2 = img.rotate(180)
    img3 = img.rotate(270)
    return [img, img1, img2, img3], [0, 1, 2, 3] # images, labels

model = resnet18()
# change the last classification layer to support 4 way classification
model.fc = nn.Linear(512,4) 

if __name__ == '__main__':
    image = Image.open("n02107683_Bernese_mountain_dog.jpeg")
    images, labels = make_example(image)
    # convert images to pytorch tensors
    images = [T.to_tensor(img) for img in images]
    labels = torch.LongTensor(labels)
    # stack images into a batch of size 4
    batch = torch.stack(images, 0)
    torchvision.utils.save_image(batch, "output/images.png", normalize=True)
    # pass the batch of rotated images from model
    logits = model(batch)
    confidence, predictions = torch.max(logits.softmax(1),1)
    for i in range(4):
        print("Network prediction for image rotated by {} degree is {} with probability  {:.3f}".format(i*90, predictions[i].item(), confidence[i].item()))
    
    loss = torch.nn.functional.cross_entropy(logits, labels)
    print("Network Loss is : ", loss.item())

### Solving jigsaw puzzles

In [ ]:
import numpy as np
from PIL import Image
import  torchvision.transforms as transforms
import torchvision
import torch

# utility function to perform RGB jittering.
def rgb_jittering(image):
    image = np.array(image, 'int32')
    # add random noise to each image channel.
    for ch in range(3):
        image[:, :, ch] += np.random.randint(-2, 2)
    image = np.clip(image, 0, 255)
    return Image.fromarray(image.astype('uint8'))

def make_example(img):
    # load a set of 1000 permuatations
    all_perm = np.load('permutations_1000.npy')

    # define an augmentation to augment a single tile of jigsaw puzzle
    augment_tile = transforms.Compose([
            transforms.RandomCrop(64),
            transforms.Resize((75, 75)),
            transforms.Lambda(rgb_jittering),
            transforms.ToTensor()])
    
    # resize image to 225 x 225 resolution
    img = img.resize((225,225))

    tile_size = 225/3 # for 3x3 jigsaw puzzle
    tiles = []
    for i in range(3):
        for j in range(3):
            # crop a tile, randomly augment it and append to a list 
            tile = img.crop([i*tile_size, j*tile_size, (i+1)*tile_size, (j+1)*tile_size])
            tile = augment_tile(tile)
            tiles.append(tile)
    
    # select a permutation from the set
    label = np.random.randint(1000)
    permutation = all_perm[label]
    # apply the permuation to the tiles
    permuted_tiles = [tiles[p] for p in permutation]    
    # return permuted tiles and permuation label
    return torch.stack(permuted_tiles, 0), label

image = Image.open("n02107683_Bernese_mountain_dog.jpeg")
image.save("output/image.png")
permuted_tiles, label = make_example(image)
torchvision.utils.save_image(permuted_tiles, "output/jigsaw.png", nrow=3, padding=0)
print("Pseudo Label is: ", label)